# Update Dates
#### Added 10 Years

## Connection to **Spark**

In [17]:
import os
import sys
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

# 1. Set PYSPARK_SUBMIT_ARGS to match your working batch file launcher
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = "C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

# staging_table_name = "staging.Integration.employee_Staging"
# wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
# spark.sql("SHOW CATALOGS").show(truncate=False)
# spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
# spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
# spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    # print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|dimension|payment_method  |false      |
|dimension|supplier        |false      |
|dimension|city            |false      |
|dimension|stock_item      |false      |
|dimension|customer        |false      |
|dimension|date            |false      |
|dimension|transaction_type|false      |
|dimension|employee        |false      |
+---------+----------------+-----------+

+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|fact     |purchase     |false      |
|fact     |stock_holding|false      |
|fact     |order        |false      |
|fact     |movement     |false      |
|fact     |sale         |false      |
|fact     |transaction  |false      |
+---------+-------------+-----------+

+-----------+-----------------------+-----------+
|namespace  |tableName              |isTemporary|
+-----------+-------------

# Update Days

## Get Max Date of Fact Sale

In [26]:
from datetime import datetime
import pyspark.sql.functions as sf

# 1. Fetch max date
max_date_rows = (
    spark.table("reporting.fact.order")
    .select(sf.max("Order_Date_Key").alias("Max_Order_Date"))
    .collect()
)

max_date_val = max_date_rows[0]["Max_Order_Date"]

# 2. Subtract matched date objects
current_date = datetime.now().date()

# If max_date_val is a datetime, ensure it converts to date
if isinstance(max_date_val, datetime):
    max_date_val = max_date_val.date()

days_diff = (current_date - max_date_val).days

print(f"Max Order Date Key: {max_date_val}")
print(f"Days Difference: {days_diff} days")

Max Order Date Key: 2026-07-28
Days Difference: 0 days


## Add Day to make Data Current

In [25]:
spark.sql(f"""
UPDATE reporting.dimension.date
SET 
    Date = date_add(Date, {days_diff}),
    Day_Number = day(date_add(Date, {days_diff})),
    Day = date_format(date_add(Date, {days_diff}), 'EEEE'),
    Month = date_format(date_add(Date, {days_diff}), 'MMMM'),
    Short_Month = date_format(date_add(Date, {days_diff}), 'MMM'),
    Calendar_Month_Number = month(date_add(Date, {days_diff})),
    Calendar_Month_Label = date_format(date_add(Date, {days_diff}), 'yyyy-MMM'),
    Calendar_Year = year(date_add(Date, {days_diff})),
    Calendar_Year_Label = concat('CY', year(date_add(Date, {days_diff}))),
    ISO_Week_Number = weekofyear(date_add(Date, {days_diff}))
""")

spark.sql(f"""
UPDATE reporting.fact.transaction
SET 
    Date_Key = date_add(Date_Key, {days_diff})
""")

spark.sql(f"""
UPDATE reporting.fact.movement
SET 
    Date_Key = date_add(Date_Key, {days_diff})
""")

spark.sql(f"""
UPDATE reporting.fact.order
SET 
    Order_Date_Key = date_add(Order_Date_Key, {days_diff}),
    Picked_Date_Key = date_add(Picked_Date_Key, {days_diff})
""")

spark.sql(f"""
UPDATE reporting.fact.purchase
SET 
    Date_Key = date_add(Date_Key, {days_diff})
""")

spark.sql(f"""
UPDATE reporting.fact.sale
SET 
    Invoice_Date_Key = date_add(Invoice_Date_Key, {days_diff}),
    Delivery_Date_Key = date_add(Delivery_Date_Key, {days_diff})
""")

# Max Date  = 07/27/2026

DataFrame[]

In [27]:
spark.stop()